# Golo v2: cifrar un lote y abrirlo

Formato v2 `global(cdn_pass + cdn)`: UN solo PRBX con el **pass global**
sobre pass pegada + contenido: `PRBX(maestro, [LOTE][u32 len][pass][datos])`.
Con el global salen el pass del lote (texto) y el archivo. v1 sigue abriendo.

In [ ]:
import os, sys
if not os.path.isdir('/tmp/golo/python/toolsec.py'):
    !git clone --depth 1 https://github.com/elmasber-ma/golo.git /tmp/golo
sys.path.insert(0, '/tmp/golo/python')
from deps import ensure
print('motor:', ensure())

In [ ]:
from toolsec import Vault

MAESTRO = 'global-demo-123'
LOTE = 'cdn-lote-7'
with open('/tmp/test.txt', 'wb') as f:
    f.write(b'hola lote v2 ' * 1000)

v = Vault(MAESTRO)
out = v.encrypt_lote_file('/tmp/test.txt', MAESTRO, LOTE, '/tmp/test.prbx')
import os as _os
print('OK lote ->', out, _os.path.getsize(out), 'bytes')

In [ ]:
from toolsec import Vault

v = Vault('global-demo-123')
r = v.decrypt_lote_file('/tmp/test.prbx', 'global-demo-123', '/tmp/test.dec')
assert r is not None, 'no abrio'
pass_lote, path = r
print('pass del lote:', pass_lote)
a = open('/tmp/test.txt', 'rb').read()
b = open(path, 'rb').read()
print('contenido igual:', a == b)

## Abrirlo en Dart

El mismo archivo abre en Dart (mimapp `crypto_vault.dart`, golo `cifre/dart/vault.dart`):

```dart
final data = await File('/tmp/test.prbx').readAsBytes();
final lote = await decryptLote(data, 'global-demo-123');
print(lote!.version);    // 2
print(lote.passLote);    // cdn-lote-7
await File('/tmp/test.dec').writeAsBytes(lote.contenido);
```